# Data Engineering

**Data Engineer: Yana** | **CAP 3764 - FIU**

This notebook collects and preprocesses historical stock price and news article data for Apple (AAPL) from 2023 to 2024.

**Imports**

In [39]:
import os
import time
import requests
import pandas as pd
import yfinance as yf
from dotenv import dotenv_values

**Load API key + config**

In [40]:
config = dotenv_values("../.env")
API_KEY = config.get("ALPHAVANTAGE_API_KEY")

print("API key loaded:", bool(API_KEY))
assert API_KEY, "ALPHAVANTAGE_API_KEY not found in ../.env"

TICKER = "AAPL"
START = "2023-01-01"
END = "2024-12-31"

ALPHA_URL = "https://www.alphavantage.co/query"

ARTICLES_OUTPUT = os.path.join("..", "data", "articles.csv")
PRICES_OUTPUT = os.path.join("..", "data", "prices.csv")

API key loaded: True


**Create monthly windows**

In [41]:
def generate_monthly_windows(start_date, end_date):
    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)

    windows = []
    current = start.replace(day=1)

    while current <= end:
        next_month = current + pd.offsets.MonthBegin(1)
        month_end = min(next_month - pd.Timedelta(days=1), end)
        month_start = max(current, start)

        windows.append((month_start, month_end))
        current = next_month

    return windows

windows = generate_monthly_windows(START, END)
print("Total windows:", len(windows))
print(windows[:3])

Total windows: 24
[(Timestamp('2023-01-01 00:00:00'), Timestamp('2023-01-31 00:00:00')), (Timestamp('2023-02-01 00:00:00'), Timestamp('2023-02-28 00:00:00')), (Timestamp('2023-03-01 00:00:00'), Timestamp('2023-03-31 00:00:00'))]


**Function to pull one month of Alpha Vantage news**

In [43]:
def fetch_news_window(ticker, start_dt, end_dt, api_key, limit=1000):
    params = {
        "function": "NEWS_SENTIMENT",
        "tickers": ticker,
        "time_from": pd.to_datetime(start_dt).strftime("%Y%m%dT0000"),
        "time_to": pd.to_datetime(end_dt).strftime("%Y%m%dT2359"),
        "sort": "EARLIEST",
        "limit": limit,
        "apikey": api_key,
    }

    response = requests.get(ALPHA_URL, params=params, timeout=60)
    data = response.json()

    if "Note" in data:
        print("Rate limit hit:", data["Note"])
        return []

    if "Information" in data:
        print("Info:", data["Information"])
        return []

    rows = []
    for item in data.get("feed", []):
        published = item.get("time_published", "")
        published_dt = pd.to_datetime(published, format="%Y%m%dT%H%M%S", errors="coerce")

        rows.append({
            "ticker": ticker,
            "date": published_dt.strftime("%Y-%m-%d") if pd.notna(published_dt) else None,
            "headline": (item.get("title") or "").strip(),
            "content": (item.get("summary") or "").strip(),
            "source": (item.get("source") or "").strip(),
            "url": (item.get("url") or "").strip(),
            "overall_sentiment_score": item.get("overall_sentiment_score"),
            "overall_sentiment_label": item.get("overall_sentiment_label"),
        })

    return rows

**Test just one month first**

In [44]:
test_rows = fetch_news_window(
    ticker=TICKER,
    start_dt="2023-01-01",
    end_dt="2023-01-31",
    api_key=API_KEY,
    limit=200
)

print("Test rows:", len(test_rows))
pd.DataFrame(test_rows).head()

Test rows: 50


,ticker,date,headline,content,source,url,overall_sentiment_score,overall_sentiment_label
0,AAPL,2023-01-02,What is TSMC?,Taiwan Semiconductor Manufacturing Co (TSMC) i...,Tech Monitor,https://www.techmonitor.ai/what-is/what-is-tsmc/,0.288280,Somewhat-Bullish
1,AAPL,2023-01-03,Apple’s market value drops below $2 trillion,Apple Inc.'s market capitalization fell below ...,Al Jazeera,https://www.aljazeera.com/economy/2023/1/3/app...,-0.167884,Somewhat-Bearish
2,AAPL,2023-01-03,If You Invested $100 in Berkshire Hathaway in ...,This article illustrates the phenomenal return...,The Globe and Mail,https://www.theglobeandmail.com/investing/mark...,0.182055,Somewhat-Bullish
3,AAPL,2023-01-03,If You Invested $100 in Berkshire Hathaway in ...,If an investor had put $100 into Berkshire Hat...,The Motley Fool,https://www.fool.com/investing/2023/01/03/if-i...,0.168116,Somewhat-Bullish
4,AAPL,2023-01-03,Foxconn to use Nvidia chips to build self-driv...,Electronics manufacturer Foxconn and chipmaker...,Reuters,https://www.reuters.com/business/autos-transpo...,0.227116,Somewhat-Bullish


**Pull all 24 months safely**

In [45]:
all_rows = []

for i, (window_start, window_end) in enumerate(windows, start=1):
    print(f"Window {i}/{len(windows)}: {window_start.date()} to {window_end.date()}")

    rows = fetch_news_window(
        ticker=TICKER,
        start_dt=window_start,
        end_dt=window_end,
        api_key=API_KEY,
        limit=1000
    )

    all_rows.extend(rows)

    # keep this to stay safe on free tier
    time.sleep(15)

articles_raw = pd.DataFrame(all_rows)
print("Raw article rows:", len(articles_raw))
articles_raw.head()

Window 1/24: 2023-01-01 to 2023-01-31
Window 2/24: 2023-02-01 to 2023-02-28
Window 3/24: 2023-03-01 to 2023-03-31
Window 4/24: 2023-04-01 to 2023-04-30
Window 5/24: 2023-05-01 to 2023-05-31
Window 6/24: 2023-06-01 to 2023-06-30
Window 7/24: 2023-07-01 to 2023-07-31
Window 8/24: 2023-08-01 to 2023-08-31
Window 9/24: 2023-09-01 to 2023-09-30
Window 10/24: 2023-10-01 to 2023-10-31
Window 11/24: 2023-11-01 to 2023-11-30
Window 12/24: 2023-12-01 to 2023-12-31
Window 13/24: 2024-01-01 to 2024-01-31
Window 14/24: 2024-02-01 to 2024-02-29
Window 15/24: 2024-03-01 to 2024-03-31
Window 16/24: 2024-04-01 to 2024-04-30
Window 17/24: 2024-05-01 to 2024-05-31
Window 18/24: 2024-06-01 to 2024-06-30
Window 19/24: 2024-07-01 to 2024-07-31
Window 20/24: 2024-08-01 to 2024-08-31
Window 21/24: 2024-09-01 to 2024-09-30
Window 22/24: 2024-10-01 to 2024-10-31
Window 23/24: 2024-11-01 to 2024-11-30
Window 24/24: 2024-12-01 to 2024-12-31
Raw article rows: 1179


,ticker,date,headline,content,source,url,overall_sentiment_score,overall_sentiment_label
0,AAPL,2023-01-02,What is TSMC?,Taiwan Semiconductor Manufacturing Co (TSMC) i...,Tech Monitor,https://www.techmonitor.ai/what-is/what-is-tsmc/,0.279867,Somewhat-Bullish
1,AAPL,2023-01-03,Apple’s market value drops below $2 trillion,Apple Inc.'s market capitalization fell below ...,Al Jazeera,https://www.aljazeera.com/economy/2023/1/3/app...,-0.184148,Somewhat-Bearish
2,AAPL,2023-01-03,If You Invested $100 in Berkshire Hathaway in ...,This article illustrates the phenomenal return...,The Globe and Mail,https://www.theglobeandmail.com/investing/mark...,0.197072,Somewhat-Bullish
3,AAPL,2023-01-03,If You Invested $100 in Berkshire Hathaway in ...,If an investor had put $100 into Berkshire Hat...,The Motley Fool,https://www.fool.com/investing/2023/01/03/if-i...,0.184606,Somewhat-Bullish
4,AAPL,2023-01-03,Foxconn to use Nvidia chips to build self-driv...,Electronics manufacturer Foxconn and chipmaker...,Reuters,https://www.reuters.com/business/autos-transpo...,0.240186,Somewhat-Bullish


**Clean articles**

In [46]:
def clean_articles(df):
    if df.empty:
        return df

    articles = df.copy()
    articles["date"] = pd.to_datetime(articles["date"], errors="coerce").dt.strftime("%Y-%m-%d")
    articles = articles.dropna(subset=["ticker", "date"])

    for col in ["headline", "content", "source", "url", "overall_sentiment_label"]:
        if col in articles.columns:
            articles[col] = articles[col].fillna("").astype(str).str.strip()

    if "url" in articles.columns:
        articles = articles.drop_duplicates(subset=["url"], keep="first")

    articles = articles.drop_duplicates(subset=["ticker", "date", "headline"], keep="first")
    articles = articles.sort_values(["date", "headline"]).reset_index(drop=True)

    keep_cols = [
        "ticker", "date", "headline", "content", "source",
        "overall_sentiment_score", "overall_sentiment_label"
    ]
    return articles[keep_cols]

articles = clean_articles(articles_raw)

print("Cleaned article rows:", len(articles))
print("Article date range:", articles["date"].min(), "to", articles["date"].max())
articles.head()

Cleaned article rows: 1167
Article date range: 2023-01-02 to 2024-12-31


,ticker,date,headline,content,source,overall_sentiment_score,overall_sentiment_label
0,AAPL,2023-01-02,What is TSMC?,Taiwan Semiconductor Manufacturing Co (TSMC) i...,Tech Monitor,0.279867,Somewhat-Bullish
1,AAPL,2023-01-03,Apple’s market value drops below $2 trillion,Apple Inc.'s market capitalization fell below ...,Al Jazeera,-0.184148,Somewhat-Bearish
2,AAPL,2023-01-03,Foxconn to use Nvidia chips to build self-driv...,Electronics manufacturer Foxconn and chipmaker...,Reuters,0.240186,Somewhat-Bullish
3,AAPL,2023-01-03,If You Invested $100 in Berkshire Hathaway in ...,This article illustrates the phenomenal return...,The Globe and Mail,0.197072,Somewhat-Bullish
4,AAPL,2023-01-03,Tuesday's ETF with Unusual Volume: IWL,The iShares Russell Top 200 ETF (IWL) experien...,Nasdaq,-0.117575,Neutral


**Save articles**

In [47]:
articles.to_csv(ARTICLES_OUTPUT, index=False, encoding="utf-8")
print(f"Saved articles to {ARTICLES_OUTPUT}")

Saved articles to ..\data\articles.csv


**Pull AAPL prices with yfinance**

In [50]:
prices = yf.download("AAPL", start="2023-01-01", end="2025-01-01", progress=False)

prices = prices.reset_index()

# Flatten columns if needed
if isinstance(prices.columns, pd.MultiIndex):
    prices.columns = [col[0] if isinstance(col, tuple) else col for col in prices.columns]

print("Columns BEFORE rename:", prices.columns)

# Rename what exists
prices = prices.rename(columns={
    "Date": "date",
    "Open": "open",
    "High": "high",
    "Low": "low",
    "Close": "close",
    "Adj Close": "adj_close",  # might not exist
    "Volume": "volume",
})

# If adj_close is missing, just use close
if "adj_close" not in prices.columns:
    prices["adj_close"] = prices["close"]

prices["date"] = pd.to_datetime(prices["date"], errors="coerce").dt.strftime("%Y-%m-%d")
prices["ticker"] = "AAPL"

prices = prices[["ticker", "date", "open", "high", "low", "close", "adj_close", "volume"]]

print("Final columns:", prices.columns)
prices.head()

Columns BEFORE rename: Index(['Date', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='object')
Final columns: Index(['ticker', 'date', 'open', 'high', 'low', 'close', 'adj_close',
       'volume'],
      dtype='object')


,ticker,date,open,high,low,close,adj_close,volume
0,AAPL,2023-01-03,128.223785,128.833995,122.210219,123.096016,123.096016,112117500
1,AAPL,2023-01-04,124.887295,126.629364,123.105865,124.365662,124.365662,89113600
2,AAPL,2023-01-05,125.123513,125.753411,122.790923,123.046814,123.046814,80962700
3,AAPL,2023-01-06,124.021172,128.233612,122.918847,127.574188,127.574188,87754700
4,AAPL,2023-01-09,128.410812,131.304413,127.839965,128.095856,128.095856,70790800


**Save prices**

In [51]:
prices.to_csv(PRICES_OUTPUT, index=False, encoding="utf-8")
print(f"Saved prices to {PRICES_OUTPUT}")

Saved prices to ..\data\prices.csv


**Validation summary**

In [52]:
print("===== VALIDATION SUMMARY =====")

print("\nArticles shape:", articles.shape)
print("Prices shape:", prices.shape)

print("\nMissing values in articles:")
print(articles.isna().sum())

print("\nMissing values in prices:")
print(prices.isna().sum())

print("\nDuplicate rows in articles:", articles.duplicated().sum())
print("Duplicate rows in prices:", prices.duplicated().sum())

===== VALIDATION SUMMARY =====

Articles shape: (1167, 7)
Prices shape: (502, 8)

Missing values in articles:
ticker                     0
date                       0
headline                   0
content                    0
source                     0
overall_sentiment_score    0
overall_sentiment_label    0
dtype: int64

Missing values in prices:
ticker       0
date         0
open         0
high         0
low          0
close        0
adj_close    0
volume       0
dtype: int64

Duplicate rows in articles: 0
Duplicate rows in prices: 0


**Optional small EDA**

In [53]:
articles_eda = articles.copy()
articles_eda["date"] = pd.to_datetime(articles_eda["date"])
articles_eda["year"] = articles_eda["date"].dt.year

print("Articles per year:")
print(articles_eda["year"].value_counts().sort_index())

prices_eda = prices.copy()
prices_eda["date"] = pd.to_datetime(prices_eda["date"])
prices_eda["year"] = prices_eda["date"].dt.year

print("\nTrading days per year:")
print(prices_eda["year"].value_counts().sort_index())

Articles per year:
year
2023    524
2024    643
Name: count, dtype: int64

Trading days per year:
year
2023    250
2024    252
Name: count, dtype: int64


**Aggregate sentiment per day**

In [54]:
articles["date"] = pd.to_datetime(articles["date"])

daily_sentiment = (
    articles.groupby("date", as_index=False)["overall_sentiment_score"]
    .mean()
    .rename(columns={"overall_sentiment_score": "avg_sentiment"})
)

daily_sentiment.head()

,date,avg_sentiment
0,2023-01-02,0.279867
1,2023-01-03,0.033884
2,2023-01-04,-0.029135
3,2023-01-05,0.188779
4,2023-01-06,0.217019


**Prepare prices**

In [55]:
prices["date"] = pd.to_datetime(prices["date"])

**Merge**

In [56]:
dataset = prices.merge(daily_sentiment, on="date", how="left")

**Fill missing sentiment**

In [57]:
dataset["avg_sentiment"] = dataset["avg_sentiment"].fillna(0)

**Create target**

In [58]:
dataset["next_close"] = dataset["close"].shift(-1)

dataset["target"] = (dataset["next_close"] > dataset["close"]).astype(int)

**Add basic features (optional)**

In [59]:
dataset["daily_return"] = dataset["close"].pct_change()
dataset["ma_5"] = dataset["close"].rolling(5).mean()
dataset["ma_10"] = dataset["close"].rolling(10).mean()

**Clean final dataset**

In [60]:
dataset = dataset.dropna().reset_index(drop=True)

dataset.head()

,ticker,date,open,high,low,close,adj_close,volume,avg_sentiment,next_close,target,daily_return,ma_5,ma_10
0,AAPL,2023-01-17,132.701952,135.123117,132.013004,133.794434,133.794434,63646600,0.342334,133.075989,0,0.008756,131.556342,128.396024
1,AAPL,2023-01-18,134.660579,136.422321,132.898822,133.075989,133.075989,69672800,-0.010855,133.135025,1,-0.005370,132.438205,129.394022
2,AAPL,2023-01-19,131.963805,134.099553,131.658700,133.135025,133.135025,58280400,-0.038847,135.693985,1,0.000444,132.788580,130.270958
3,AAPL,2023-01-20,133.144867,135.841627,132.101599,135.693985,135.693985,80223600,0.000000,138.882874,1,0.019221,133.666498,131.535675
4,AAPL,2023-01-23,135.940059,141.058000,135.723530,138.882874,138.882874,81760300,0.297545,140.280441,1,0.023501,134.916461,132.666544


**Save final dataset**

In [61]:
dataset.to_csv("../data/final_dataset.csv", index=False)